In [ ]:
pip install tensorflow mlflow

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,LSTM, Dense, Dropout, Bidirectional, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
# Set random seeds for reproducibility
np.random.seed(123)
tf.random.set_seed(123)
import joblib
import json 
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

import mlflow
import plotly.express as px




from keras.layers import Input, Dropout, Dense, LSTM, TimeDistributed, RepeatVector
from keras.models import Model
from keras import regularizers

In [ ]:
df_r1 = pd.read_parquet("prepared_data_for_ml_ro1.parquet")

In [ ]:
def feature_engg(plc):
    plc['pi_04_05_diff'] = plc['PI-04']- plc['PI-05']
    plc['ti04_08_diff'] = plc['TI-04']- plc['TI-08']
    return plc
df_r1 = feature_engg(df_r1)

In [ ]:
df_r1.columns

In [ ]:
imp_cols = ['batch_time_r1_mins','DPI-01', 'FI_DEN-02', 'FI_DEN-04', 'ti04_08_diff']

In [ ]:
train_df = df_r1[df_r1['Step-04 (98.5 %) r1']>98.51]

In [ ]:
train_df = train_df.sort_values(by = ['batch_id_r1', 'batch_time_r1_mins'])

In [ ]:
train_df.dropna(subset=imp_cols, inplace=True)

In [ ]:
train_df.isnull().sum()

In [ ]:
import numpy as np
import pandas as pd

def create_lstm_data_next_step(
    df,
    feature_cols,
    target_col='quality',
    batch_col='batch_id',
    time_col='batch_time',
    window_size=30,
    stride=1  # usually 1 for next-step prediction
):
    """
    Prepare LSTM sequences for predicting the next-step quality.
    
    Args:
        df: pandas DataFrame with batch, time, features, target
        feature_cols: list of feature column names
        target_col: name of the target column
        batch_col: column identifying batch
        time_col: column identifying time in minutes
        window_size: number of past minutes to use as input
        stride: how many minutes to move the window each step
    
    Returns:
        X: numpy array of shape (num_sequences, window_size, num_features)
        y: numpy array of next-step target values
        batch_list: batch ID for each sequence
    """
    X = []
    y = []
    batch_list = []

    # Sort dataframe by batch and time
    df = df.sort_values([batch_col, time_col]).reset_index(drop=True)

    # Loop over each batch
    for batch_id, g in df.groupby(batch_col):
        g = g.sort_values(time_col).reset_index(drop=True)

        values = g[feature_cols].values
        times = g[time_col].values

        # Slide window over batch
        for start in range(0, len(g) - window_size):
            end = start + window_size
            window_times = times[start:end]

            # Ensure continuous time (minute-level)
            if np.all(np.diff(window_times) == 1):
                X.append(values[start:end])
                y.append(g[target_col].iloc[end])  # next-step target
                batch_list.append(batch_id)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)
    batch_list = np.array(batch_list)

    print(f"\nCreated {len(X)} sequences for next-step prediction")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"Target ({target_col}) range: [{y.min():.2f}, {y.max():.2f}]")

    return X, y, batch_list

In [ ]:
X, y , batch_list = create_lstm_data_next_step(
    train_df,
    imp_cols,
    target_col='Step-04 (98.5 %) r1',
    batch_col='batch_id_r1',
    time_col='batch_time_r1_mins',
    window_size=60,
    stride=1
)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Reshape to 2D for scaling (num_samples*sequence_length, n_features)
num_samples, seq_len, n_features = X.shape
scaler = StandardScaler()

X_train_2d = X.reshape(-1, n_features)
X_train_scaled = scaler.fit_transform(X_train_2d).reshape(num_samples, seq_len, n_features)

In [ ]:
X_train_scaled.shape, X.shape

In [ ]:
def autoencoder_model(X):
    inputs = Input(shape=(X.shape[1], X.shape[2]))

    L1 = LSTM(64, activation='relu', return_sequences=True)(inputs)
    L2 = LSTM(32, activation='relu', return_sequences=True)(L1)
    L3 = LSTM(8, activation='relu', return_sequences=True)(L2)
    L4 = LSTM(4, activation='relu', return_sequences=False)(L3)

    L5 = RepeatVector(X.shape[1])(L4)
    L6 = LSTM(4, activation='relu', return_sequences=True)(L5)
    L7 = LSTM(16, activation='relu', return_sequences=True)(L6)

    output = TimeDistributed(Dense(X.shape[2]))(L7)

    model = Model(inputs=inputs, outputs=output)
    return model

In [ ]:
model = autoencoder_model(X_train_scaled)
model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train_scaled, X_train_scaled,
    validation_split=0.06, 
    epochs=300,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

def plot_training_validation_loss(history):
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'], label='Train Loss')
    
    if 'val_loss' in history.history:
        plt.plot(history.history['val_loss'], label='Validation Loss')
    
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

# call
plot_training_validation_loss(history)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reconstruct train data
X_train_pred = model.predict(X_train_scaled)

# Reconstruction error per training sample
# shape after mean: (num_samples,)
train_mae_loss = np.mean(np.abs(X_train_pred - X_train_scaled), axis=(1, 2))
train_mse_loss = np.mean(np.square(X_train_pred - X_train_scaled), axis=(1, 2))

print("Train MAE error shape:", train_mae_loss.shape)
print("Train MSE error shape:", train_mse_loss.shape)

In [ ]:
import joblib

# Save model
model.save("lstm_autoencoder_model.keras")   # recommended format

# Save scaler
joblib.dump(scaler, "standard_scaler.pkl")

print("Model and scaler saved successfully.")

In [ ]:
def plot_train_errors(errors, title='Train Reconstruction Error'):
    plt.figure(figsize=(12, 5))
    plt.plot(errors, label='Reconstruction Error')
    plt.xlabel('Training Sample Index')
    plt.ylabel('Error')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

# Plot MAE
plot_train_errors(train_mae_loss, title='Train Reconstruction Error (MAE)')

# Plot MSE
plot_train_errors(train_mse_loss, title='Train Reconstruction Error (MSE)')

In [ ]:
threshold = np.mean(train_mae_loss) + 3 * np.std(train_mae_loss)

plt.figure(figsize=(12, 5))
plt.plot(train_mae_loss, label='Train Reconstruction Error')
plt.axhline(threshold, linestyle='--', label=f'Threshold = {threshold:.4f}')
plt.xlabel('Training Sample Index')
plt.ylabel('MSE Error')
plt.title('Train Reconstruction Error with Threshold')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import pandas as pd

def create_lstm_data_with_indices(
    df,
    feature_cols,
    target_col,
    batch_col,
    time_col,
    window_size=60,
    stride=1
):
    X = []
    y = []
    batch_list = []
    row_indices = []

    df = df.sort_values([batch_col, time_col]).reset_index()

    for batch_id, g in df.groupby(batch_col):
        g = g.sort_values(time_col).reset_index(drop=True)

        values = g[feature_cols].values
        times = g[time_col].values
        original_idx = g['index'].values

        for start in range(0, len(g) - window_size, stride):
            end = start + window_size
            window_times = times[start:end]

            if np.all(np.diff(window_times) == 1):
                X.append(values[start:end])
                y.append(g[target_col].iloc[end])
                batch_list.append(batch_id)
                row_indices.append(original_idx[end])   # map anomaly to the predicted row

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)
    batch_list = np.array(batch_list)
    row_indices = np.array(row_indices)

    return X, y, batch_list, row_indices

In [ ]:
test_df, y , batch_list , row_indices = create_lstm_data_with_indices(
    df_r1,
    imp_cols,
    target_col='Step-04 (98.5 %) r1',
    batch_col='batch_id_r1',
    time_col='batch_time_r1_mins',
    window_size=60,
    stride=1
)

In [ ]:
num_samples, seq_len, n_features = test_df.shape

test_df_2d = test_df.reshape(-1, n_features)
test_df_scaled = scaler.transform(test_df_2d).reshape(num_samples, seq_len, n_features)

In [ ]:
pred_test = model.predict(test_df_scaled)

In [ ]:
main_errors = np.mean(np.square(pred_test - test_df_scaled), axis=(1, 2))

main_errors_abs = np.mean(np.abs(pred_test - test_df_scaled), axis=(1, 2))
threshold = np.mean(train_mae_loss) + 2.5 * np.std(train_mae_loss)
threshold

In [ ]:
main_anomaly_flag = (main_errors_abs > threshold).astype(int)

In [ ]:
df_r1_result = df_r1.copy()
df_r1_result['reconstruction_error'] = np.nan
df_r1_result['anomaly'] = 0

df_r1_result.loc[row_indices, 'reconstruction_error'] = main_errors
df_r1_result.loc[row_indices, 'anomaly'] = main_anomaly_flag

In [ ]:
px.line(df_r1_result[df_r1_result['batch_id_r1']==95], x = 'batch_time_r1_mins', y = 'FI_DEN-04', color='anomaly')

In [ ]:
bmr = pd.read_excel('Sampling_data.xlsx')

In [ ]:
plc = df_r1_result.copy()


In [ ]:
plc

In [ ]:
batch_dict = {'#116': 90,
#  '#123': 93,
#  '#129': 96,
#  '#137': 100,
#  '#003': 116,
#  '#009': 119,
#  '#015': 122,
#  '#032': 130,
#  '#037': 133,
#  '#043': 136,
 '#048 (RP)': 140,
 '#052': 142,
 '#058 (RP)': 145,
 '#065': 148,
 '#070': 151,
 
 '#075': 154}

In [ ]:
for key, values in batch_dict.items():
    print(f"Plotting for batch {key} and created batch_id {values}")

    df_temp = plc[plc['batch_id_r1'] == values].copy()
    # df_temp = df_temp[(df_temp['DPI-01'] > 0)&(df_temp['FI_DEN-04'].between(1550,1620))]

    sample_df = bmr[
        (bmr['Equipment No.'] == 'R-01') &
        (bmr['Batch No.'].isin([key]))
    ][['Batch No.', 'Sampling Date', 'Sampling Time', 'Step-04 (98.5 %)']].copy()


    sample_df["sample_dt"] = pd.to_datetime(
            pd.to_datetime(sample_df["Sampling Date"]).dt.strftime("%Y-%m-%d") + " " +
            sample_df["Sampling Time"].astype(str)
        )

    sample_df["Step-04 (98.5 %)"] = pd.to_numeric(sample_df["Step-04 (98.5 %)"], errors="coerce")

    df_temp["DateTime"] = pd.to_datetime(df_temp["DateTime"])
    df_temp = df_temp.sort_values("DateTime")

    sample_match = pd.merge_asof(
        sample_df.sort_values("sample_dt"),
        df_temp[["DateTime", "batch_time_r1_mins"]].sort_values("DateTime"),
        left_on="sample_dt",
        right_on="DateTime",
        direction="nearest",
        tolerance=pd.Timedelta("5min")
    )

    sample_match = sample_match.dropna(subset=["batch_time_r1_mins"])

    sample_cols = ['FI_DEN-04', 'DPI-01']

    for col in sample_cols:
        fig = px.line(
            df_temp,
            x="batch_time_r1_mins",
            y=col,
            color='anomaly',
            hover_data=["DateTime"]
        )

        for _, row in sample_match.iterrows():
            fig.add_vline(
                x=row["batch_time_r1_mins"],
                line_dash="solid" if row["Step-04 (98.5 %)"] > 98.5 else "dot",
                line_width=1.5 
            )
        fig.show()